# 极化指标包入门（AP-1）

消费 `rfauto.core.polarization`（轴比/旋向/圆极化判别/极化失配，Balanis 4e §4.4 + IEEE Std 145 口径）：单点极化态、闭式 vs 时域数值双路径互证、经典 PLF 锚。

- 纯函数零 IO；旋向约定=沿传播方向看顺时针为右旋（IEEE）；
- 线极化 AR→∞ 按 100 dB 封顶（量纲上等价发散）。

In [ ]:
from rfauto.core.polarization import (
    axial_ratio,
    axial_ratio_db,
    axial_ratio_time_domain,
    classify_cp,
    polarization_loss_factor,
    polarization_state,
)

lin = polarization_state(1.0 + 0j, 0.5 + 0j)
rhcp = polarization_state(1.0 + 0j, -1.0j)  # δ=-90° → 右旋
print("线极化:", {k: lin[k] for k in ("ar_db", "sense")})
print("右旋圆:", {k: rhcp[k] for k in ("ar_db", "sense")})
assert rhcp["sense"] == "RH"
assert lin["sense"] == "linear"

In [ ]:
# 闭式 vs 时域椭圆数值法互证（仓内两独立实现，#118 双路径）
import math

e1, e2, delta = 1.0, 0.6, math.radians(52.0)
ar_closed = axial_ratio(e1, e2, delta)
ar_numeric = axial_ratio_time_domain(e1, e2, delta)
print(f"闭式   AR = {ar_closed:.6f} ({axial_ratio_db(e1, e2, delta):.3f} dB)")
print(f"时域数值 AR = {ar_numeric:.6f}")
assert abs(ar_closed - ar_numeric) < 1e-3
print("分类:", classify_cp(axial_ratio_db(e1, e2, delta)))

In [ ]:
# 极化失配因子经典锚（Balanis §4.4.3）
plf_aligned = polarization_loss_factor(1, 0, 0, 1, 0, 0)
plf_ortho = polarization_loss_factor(1, 0, 0, 0, 1, 0)
plf_cp_lin = polarization_loss_factor(
    1, 1, -math.pi / 2, 1, 0, 0)  # 圆 × 线
print(f"线-线对齐 = {plf_aligned}")
print(f"线-线正交 = {plf_ortho}")
print(f"圆-线     = {plf_cp_lin} (=-3.01 dB)")
assert plf_aligned == 1.0 and plf_ortho == 0.0
assert abs(plf_cp_lin - 0.5) < 1e-12